In [19]:
import logging
import re
from typing import Literal, Annotated
import json
import numpy as np
import numpy.typing as npt
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm import tqdm
from scipy.stats import gaussian_kde
from sklearn.neighbors import KernelDensity
import seaborn as sns
from tqdm.notebook import tqdm
from IPython.display import display, Markdown

logging.basicConfig(format='%(asctime)s [%(levelname)s] %(name)s: %(message)s', level=logging.DEBUG)
logger = logging.getLogger('base')
logging.getLogger('matplotlib').setLevel(logging.WARNING)
logger.setLevel(logging.DEBUG)

pd.options.display.max_columns = 650
pd.options.display.max_rows = 20

In [10]:
path_rankings = Path('../data/rankings')
path_export = Path('../data/converted')

cached_df_res = Path('../data/df_res.csv')
RECALL_TARGETS = [0.8, 0.85, 0.9, 0.95, 0.99, 1.0]
df_res = pd.read_csv(cached_df_res)

/tmp/ipykernel_2602552/3338261287.py:6: DtypeWarning: Columns (44,45) have mixed types. Specify dtype option on import or set low_memory=False.
  df_res = pd.read_csv(cached_df_res)


In [9]:
df_res

,dataset,sim-rep,sim_key,batch_i,n_total,n_seen,n_unseen,n_incl,n_incl_seen,n_incl_batch,n_records_batch,method,safe_to_stop,method-hash,method-KEY,method-safe_to_stop,method-score,method-confidence_level,method-recall_target,method-bias,method-est_incl,method-inclusion_threshold,method-batch_size,method-threshold,method-current_precision,method-window_size,method-polyorder,method-threshold_ratio,method-threshold_peak,method-slope_ratio,method-smoothing,method-num_to_stop,method-fraction,method-sample_size,method-n_windows,method-est_recall,method-expected_includes,method-curve_estimate,method-nstd,method-est_var,method-alpha,method-constant,method-num_reviewed,method-num_relevant_reviewed,method-use_adjusted,method-use_margin,method-margin_recall,method-positive_sample_size,method-required_overlap,method-n_overlap,method-n_sample,seen@recall=0.8,unseen@recall=0.8,seen_incl@recall=0.8,too_late@recall=0.8,too_late_left%@recall=0.8,too_late_right%@recall=0.8,too_early@recall=0.8,too_early%@recall=0.8,missed@recall=0.8,missed%@recall=0.8,seen@recall=0.85,unseen@recall=0.85,seen_incl@recall=0.85,too_late@recall=0.85,too_late_left%@recall=0.85,too_late_right%@recall=0.85,too_early@recall=0.85,too_early%@recall=0.85,missed@recall=0.85,missed%@recall=0.85,seen@recall=0.9,unseen@recall=0.9,seen_incl@recall=0.9,too_late@recall=0.9,too_late_left%@recall=0.9,too_late_right%@recall=0.9,too_early@recall=0.9,too_early%@recall=0.9,missed@recall=0.9,missed%@recall=0.9,seen@recall=0.95,unseen@recall=0.95,seen_incl@recall=0.95,too_late@recall=0.95,too_late_left%@recall=0.95,too_late_right%@recall=0.95,too_early@recall=0.95,too_early%@recall=0.95,missed@recall=0.95,missed%@recall=0.95,seen@recall=0.99,unseen@recall=0.99,seen_incl@recall=0.99,too_late@recall=0.99,too_late_left%@recall=0.99,too_late_right%@recall=0.99,too_early@recall=0.99,too_early%@recall=0.99,missed@recall=0.99,missed%@recall=0.99,seen@recall=1.0,unseen@recall=1.0,seen_incl@recall=1.0,too_late@recall=1.0,too_late_left%@recall=1.0,too_late_right%@recall=1.0,too_early@recall=1.0,too_early%@recall=1.0,missed@recall=1.0,missed%@recall=1.0,stop_recall,work_saved,incl_missed,target_recall_reached,incl_rate,tetl_left@recall=0.8,tetl_right@recall=0.8,mtl_right@recall=0.8,mtl_left@recall=0.8,missed_%@recall=0.8,additional_work,tetl_right,tetl_left,mtl_left,mtl_right,missed_%,tetl_left@recall=0.85,tetl_right@recall=0.85,mtl_right@recall=0.85,mtl_left@recall=0.85,missed_%@recall=0.85,tetl_left@recall=0.9,tetl_right@recall=0.9,mtl_right@recall=0.9,mtl_left@recall=0.9,missed_%@recall=0.9,tetl_left@recall=0.95,tetl_right@recall=0.95,mtl_right@recall=0.95,mtl_left@recall=0.95,missed_%@recall=0.95,tetl_left@recall=0.99,tetl_right@recall=0.99,mtl_right@recall=0.99,mtl_left@recall=0.99,missed_%@recall=0.99,tetl_left@recall=1.0,tetl_right@recall=1.0,mtl_right@recall=1.0,mtl_left@recall=1.0,missed_%@recall=1.0
0,clef-CD005139,1,clef-CD005139-0-500-1-best,304,4564,4564,0,105,105,0,15,CURVE_FITTING,False,ALISON-417e55e2a26d7f24cad8370396678f36024acb8f,ALISON,False,NaN,0.80,0.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,121.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,900,3664,86,3664,4.071111,1.000000,0,0.0,0,0.0,960,3604,90,3604,3.754167,1.000000,0,0.0,0,0.0,1140,3424,96,3424,3.003509,1.000000,0,0.0,0,0.0,1545,3019,100,3019,1.954045,1.000000,0,0.0,0,0.0,2565,1999,104,1999,0.779337,1.000000,0,0.000000,0,0.000000,3465,1099,105,1099,0.317172,1.000000,0,0.000000,0,0.000000,1.000000,0.000000,0.000000,True,0.023006,407.111111,100.000000,100.000000,407.111111,-0.0,43.799299,100.000000,77.933723,77.933723,100.000000,-0.0,375.416667,100.000000,100.000000,375.416667,-0.0,300.350877,100.000000,100.000000,300.350877,-0.0,195.404531,100.000000,100.000000,195.404531,-0.0,77.933723,100.000000,100.000000,77.933723,-0.000000,31.717172,100.000000,100.000000,31.717172,-0.000000
1,clef-CD005139,1,clef-CD005139-0-500-1-best,230,4564,3465,1099,105,105,1,15,CURVE_FITTING,True,ALISO

In [16]:
%%bash
ls -lisah ../data/

total 79M
143531119 4.0K drwxr-xr-x 10 rept rept 4.0K Oct 23 14:36 .
139494298 4.0K drwxrwxr-x 12 rept rept 4.0K Oct 20 13:05 ..
143679054 4.0K drwxr-xr-x  2 rept rept 4.0K Jan 27  2025 code
143531126  28K -rw-r--r--  1 rept rept  26K Oct 20 19:25 dataset_selection.csv
143531128  79M -rw-rw-r--  1 rept rept  79M Oct 23 14:36 df_res.csv
143531127 4.0K drwxrwxr-x  2 rept rept 4.0K Oct 22 20:37 .ipynb_checkpoints
143679055 4.0K drwxr-xr-x  2 rept rept 4.0K May 21  2025 logs
143679056 4.0K drwxr-xr-x  4 rept rept 4.0K May 19  2025 models
143531121 4.0K -rw-r--r--  1 rept rept 1.7K Oct 17 19:31 notes.md
143679057 4.0K drwxr-xr-x  4 rept rept 4.0K Oct 31 23:34 plots
143679058  36K drwxr-xr-x  3 rept rept  36K Jan  8 20:57 rankings
143531120 4.0K drwxr-xr-x  7 rept rept 4.0K May 26  2025 raw
143676368 4.0K drwxr-xr-x  2 rept rept 4.0K Oct 22 20:57 results


In [66]:
%%bash
du -hd1 ../data

4.0K	../data/.ipynb_checkpoints
4.0G	../data/results
248K	../data/code
4.5G	../data/raw
4.0K	../data/logs
46M	../data/converted
9.7M	../data/plots
110M	../data/rankings
35M	../data/models
8.6G	../data


# Converted format

Each file contains all simulations and stopping decision for a specific dataset.

* `name`: Dataset name
* `n_total`: Overall size of dataset (number of records)
* `n_incl`: Number of relevant records
* `simulations`: Each simulation (typically, we pre-compute three rankings with different random initial set per dataset)
   * `ranking_info`: Information on which ranking model was used for each batch and how large the batches were (this is dynamic)
   * `ranking`: The ordered list of records
      * `id`: row indices in the original raw dataset for reference
      * `labels`: inclusion (1) /exclusion (0) annotations
      * `score`: model score, -1 if part of random sample
      * `batch`: which training batch this is part of
   * `stop_decisions`: For each stopping method and set of parameters
      * `method`: Name of the stopping method
      * `n_seen`: How many records were screened before the method with these `params` said stop
      * `n_incl_seen`: How many of those records were relevant

Notes:
* The stopping decisions were computed on fixed batch sizes of 15 on the pre-computed ranking
* The `params` in the `stop_decisions` are repeated and duplicates, but might be more convenient to use this way

In [65]:
# 'hash', 'KEY', 'safe_to_stop', 'score', 
param_cols = {
    f'method-{col}': col
    for col in [
        'confidence_level', 'recall_target', 'bias', 'est_incl', 'inclusion_threshold', 'batch_size', 'threshold', 'current_precision', 'window_size', 'polyorder', 'threshold_ratio', 'threshold_peak', 
        'slope_ratio', 'smoothing', 'num_to_stop', 'fraction', 'sample_size', 'n_windows', 'est_recall', 'expected_includes', 'curve_estimate', 'nstd', 'est_var', 'alpha', 'constant', 'num_reviewed', 
        'num_relevant_reviewed', 'use_adjusted', 'use_margin', 'margin_recall', 'positive_sample_size', 'required_overlap', 'n_overlap', 'n_sample']
}

for dataset, df_ds in df_res.groupby('dataset'):
    info = {
        'name': dataset.replace('generic-csv-', 'SYNERGY: ').replace('generic-paired-ris-', 'EPPI: ').replace('clef-', 'CLEF: '),
        'n_total': int(df_ds.iloc[0]['n_total']),
        'n_incl': int(df_ds.iloc[0]['n_incl']),
        'simulations': [],
    }
    for repeat, df_rep in df_ds.groupby('sim-rep'):
        simkey = df_rep.iloc[0]['sim_key']
        with open(path_rankings / f'{simkey}.json') as fp:
            rankinfo = json.load(fp)
        ranking = pd.read_feather(path_rankings / f'{simkey}.feather')

        siminfo = {
            'ranking_info': rankinfo,
            'ranking': {
                'id': ranking['id'].tolist(),
                'labels': ranking['label'].tolist(), 
                'score': ranking['score'].fillna(-1).tolist(),
                'batch': ranking['batch'].tolist(),
            },
            'stop_decisions': [],
        }

        for _, row in df_rep.replace({np.nan: None})[list(param_cols.keys()) + ['n_seen', 'n_incl_seen', 'method']].rename(columns=param_cols).sort_values('n_seen').iterrows():
            params = {k: v for k,v in row.to_dict().items() if v is not None}
            stopinfo = {
                'method': params['method'],
                'n_seen': params['n_seen'],
                'n_incl_seen': params['n_incl_seen'],
            }
            del params['n_seen']
            del params['method']
            del params['n_incl_seen']
            stopinfo['params'] = params
            siminfo['stop_decisions'].append(stopinfo)
        
        info['simulations'].append(siminfo)

    with open(path_export / f'{dataset}.json', 'w') as fp:
        json.dump(info, fp)
